[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dpadillaor/Sem3_SI_ImageEmbedding/blob/main/Colab_Seminario3.ipynb)

# **IMPORTANTE**
## **CREA UNA COPIA DE ESTE NOTEBOOK EN TU ESPACIO DE GOOGLE DRIVE.**
### Deberás entregar la copia en la tarea del seminario.

## Setup previo

Para poder ejecutar esta prate del seminario correctamente necesitaremos tener una GPU. Asegurate de que tu runtime de Colab este usando GPU tal siguiendo las indicaciones de las siguientes imagenes:

<br>
<div style="display: flex; gap: 16px; justify-content: center; flex-wrap: wrap;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/RunTime_Colab.png" width="400">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/GPU_Selecton_Colab.png" width="400">
</div>


# Seminario 3: De Píxeles a Significado Compartido
## Visión por Computador y Modelos Multimodales

Date: 22 March 2026

Authors: David Padilla Orenga

Version: 1.0

---

### Contexto

En los dos seminarios anteriores aprendiste que el **texto se convierte en vectores** y que puedes medir similitud semántica con la **similitud coseno**. Hoy vas a descubrir que las **imágenes también se pueden convertir en vectores**... y que texto e imagen pueden vivir en el **mismo espacio vectorial**.

El coseno que usabas para comparar *'banco'* y *'financiero'* va a funcionar ahora para comparar una frase y una foto.

### Pregunta que abre este seminario

> *¿Puede una máquina entender que esta foto y esta frase dicen lo mismo?*

Al final de la sesión habrás construido tú mismo la respuesta.


<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/CLIP_Example.png" width="800">
</div>

### Objetivos de aprendizaje

Al finalizar esta práctica serás capaz de:
- Entender cómo una imagen se representa como datos numéricos
- Comprender la analogía entre tokenización de texto y *patchification* de imágenes
- Generar embeddings visuales y medir similitud entre imágenes
- Entender cómo CLIP conecta texto e imagen en un espacio compartido
- Construir un clasificador zero-shot y un buscador semántico de imágenes

### Estructura de la sesión (~2 horas)

| Parte | Tema | Tiempo |
|-------|------|--------|
| Setup | Instalación y carga de modelos | 5 min |
| Hook | El truco de magia: CLIP antes de explicarlo | 10 min |
| I | Imágenes como datos: píxeles y patchification | 20 min |
| II | Embeddings visuales | 20 min |
| III | El espacio compartido texto ↔ imagen | 25 min |
| IV | Zero-shot classification y búsqueda semántica | 20 min |
| V *(opcional)* | Segmentación guiada por texto con SAM | 15 min |

---
## Configuración del entorno

Ejecuta esta celda primero. Puede tardar un par de minutos la primera vez. Este bloque instalará todas las dependencias necesarias para realizar este seminario.

In [ ]:
%%capture
!pip install transformers torch torchvision Pillow scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import os
import random

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully")

A continuación cargaremos el modelo que utilizaremos para esta sesión. En este caso usaremos **CLIP** un modelo dual de OpenAI que permite hacer *embeddings de imagenes y de texto*. Para cada caso, se usan mecanismos distintos que veremos más adelante en el seminario.

In [ ]:
!pip install open_clip_torch

## Cargando el modelo: CLIP no es un modelo, es una familia

Cuando decimos "vamos a usar CLIP", en realidad estamos tomando una
decisión de ingeniería: **¿qué modelo concreto usamos?**

CLIP es un *enfoque* de entrenamiento (texto + imagen en un espacio
compartido), pero existen decenas de implementaciones con diferencias
importantes:

- **Arquitectura**: ViT-B/32, ViT-L/14, ViT-H/14... — más grande = más
preciso, más lento y más pesado
- **Datos de entrenamiento**: OpenAI entrenó con 400M pares; OpenCLIP
re-entrenó con LAION (5B pares); SigLIP (Google) usa una función de
pérdida distinta
- **Resolución de entrada**: patch32 vs patch14 — afecta al detalle que
capta el modelo
- **Tamaño en disco**: desde ~150 MB hasta varios GB

En proyectos reales, la elección del modelo depende de tus necesidades:
¿necesitas máxima precisión o que corra en un móvil? ¿tienes GPU o solo
CPU? ¿cuánta memoria tienes disponible?

La librería `open_clip` da acceso a todos estos modelos con una API
unificada. A continuación, cuando corráis la siguiente celda, podréis ver la lista completa de modelos
disponibles:

In [ ]:
import open_clip
open_clip.list_pretrained()

## ¿Qué modelo hemos elegido y por qué?

De toda la lista anterior, usamos **`ViT-B-32` con pesos de OpenAI** — el
CLIP original. Es el modelo de referencia: suficientemente preciso para
explorar todos los conceptos del seminario, ligero (~150 MB) y rápido
incluso en CPU.

> En la Part 2 del seminario usaremos **SigLIP2**, un modelo más moderno y
  preciso. Pero para aprender los fundamentos, ViT-B-32 es perfecto.

 ---

## CPU vs GPU: ¿dónde ejecutamos el modelo?

Las redes neuronales grandes hacen millones de multiplicaciones de
matrices. Las GPUs están diseñadas exactamente para eso — pueden ser
**100x más rápidas** que una CPU para este tipo de cálculo.

El código detecta automáticamente si hay GPU disponible:
- `cuda` → hay GPU (NVIDIA) → usamos `model.cuda()` para mover el modelo a
  la GPU
- `cpu` → no hay GPU → el modelo se queda en RAM y funciona igual, solo
más lento

**Aseguraos de que el print dice `cuda`** — si no, id a `Entorno de
ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

> Todos los tensores que paséis al modelo deberán estar en el mismo
dispositivo que el modelo. Si el modelo está en GPU, los datos también
deben ir a GPU (`.to(device)`). Lo veréis en las celdas siguientes.

In [ ]:
import torch
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32",pretrained="openai")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device used: ", device)
if device == "cuda":
    model.cuda().eval()
else:
    model.eval()

## Mecanismos de preprocessamiento de texto e imagen

Ya tenemos el modelo cargado en **GPU**, el mecanismo central que convierte
cualquier entrada en un vector del espacio compartido. Pero el modelo por
sí solo no puede hacer nada: necesita que le demos los datos en el formato
exacto que espera.

Hasta ahora, cuando trabajábamos solo con texto, teníamos un único
componente extra: el **tokenizador**, que convertía texto → IDs numéricos
→ tensor.

Con CLIP necesitamos tres componentes:

| Componente | Entrada | Salida | Equivalente |
|---|---|---|---|
| **Tokenizador** | texto | tensor de IDs | igual que en Sem1/Sem2 |
| **Procesador de imagen** | imagen PIL / numpy | tensor normalizado | el "tokenizador" de imágenes |
| **Modelo** | tensores de texto o imagen | vector en espacio compartido | la "caja negra" central |
<br>

Como trabajamos con imagenes necesitamos un componente el **procesador**. El procesador de imagen es un mecanismo necesario porque cada imagen llega en un formato distinto: distinta resolución, distintos valores de píxel, distinto rango. El modelo espera siempre exactamente el mismo formato
(224×224, normalizado con unos valores concretos). El procesador se
encarga de esa estandarización, igual que el tokenizador estandariza el
texto a secuencias de 77 tokens.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/CLIP_Dual_Encoder.png" width="800">
</div>

### Mecanismos de preprocessamiento de imagen - **Procesador**

Ya cargamos `preprocess` junto al modelo al usar `create_model_and_transforms`
en la casilla de código anterior. El preprocesador está configurado específicamente para el modelo que se ha indicado.

En la siguiente celda aplicamos `preprocess` para observar todas las operaciones que se le aplicarán a una imagen cuando las pasemos por él. El *output* será un tensor `[3, 224, 224]` listo para entrar al encoder visal.



In [ ]:
preprocess

### Mecanismos de preprocesamiento de texto - Tokenizador

Solo nos falta el *tokenizador de texto*. Para el resto del seminario
usaremos directamente `open_clip.get_tokenizer('ViT-B-32')`, pero en este
bloque cargaremos también el de `transformers` — **es el mismo tokenizador
  con otra API** que nos permite inspeccionar qué está pasando por dentro:
ver los tokens como strings, decodificar IDs de vuelta a texto, y entender
  mejor cómo el modelo "lee" una frase.

Fíjate en el nombre del modelo: `clip-vit-base-patch32`. El `patch32`
indica el tamaño de los patches en los que se divide la imagen
internamente — lo veremos en detalle más adelante. Por ahora quédate con
que es una característica del modelo que afecta a cómo "lee" las imágenes.

In [ ]:
from transformers import CLIPTokenizer

# Load the pre-trained CLIP tokenizer
tokenizer = open_clip.get_tokenizer('ViT-B-32')
tokenizer_advanced = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

# Example sentence to tokenize
sentence = "This is seminar 3 of Sistemes Intel·ligents and I'm becoming a master of tokenization."

# Tokenize the sentence:
#   - tokenizer(...) returns a dictionary; we request a PyTorch tensor with return_tensors="pt"
#   - The "input_ids" tensor holds the token IDs
tokenized_output = tokenizer_advanced(sentence, return_tensors="pt")
token_ids = tokenized_output["input_ids"][0].tolist()  # Convert tensor to a list of IDs

print("Token IDs:", token_ids)

# Convert token IDs back to tokens (subword units)
tokens = tokenizer_advanced.convert_ids_to_tokens(token_ids)
print("Tokens:", tokens)

print("Context window size:", tokenizer_advanced.model_max_length)


### Preguntas sobre el tokenizador

Vamos con unas preguntas para ver si os acordáis de los sminarios anteriores. Este modelo de tokenizador ya es un modelo más complejo que los de los seminarios anteriores. Es un modelo muy usado en la industria.

---

**P1: ¿Qué crees que quieren decir los tokens 49406 y 49407?**

*Escribe tu respuesta aquí*

---

**P2: ¿Qué crees que significa `</w>`?**

*Escribe tu respuesta aquí*

---

**P3: ¿Qué crees que significa que el contexto máximo del tokenizador es de 77?**

*Escribe tu respuesta aquí*

---

## Dataset: Caltech-101

Para los ejercicios de este seminario necesitamos un conjunto de imágenes
con las que experimentar. Usaremos **Caltech-101**, un dataset clásico de
visión por computador creado por el California Institute of Technology.

Caltech-101 contiene imágenes organizadas en 101 categorías: desde
animales y vehículos hasta instrumentos musicales, caras o sillas. Las
imágenes son fotografías reales con resolución natural, lo que hace que
los experimentos sean visualmente intuitivos.

Trabajaremos con un subconjunto de ~1.000 imágenes — 10 por cada una de
las 101 clases — lo que lo hace manejable sin perder variedad.

### Descargar el subset
 El subset está disponible en Google Drive. La siguiente celda lo descarga
  automáticamente y lo descomprime, no necesitas subir nada manualmente.

In [ ]:
!gdown "https://drive.google.com/uc?id=1pfqPQo0xQV9xZ9Q84q838wIzeFmXtTyY" -O Caltech101_subset.zip
!unzip -q Caltech101_subset.zip -d Caltech101_subset

Dentro de `Caltech101_subset` encontrarás 102 carpetas: una por cada clase (101 categorías + 1 carpeta de background) con 10 imágenes cada una. Esta
estructura es la estándar en datasets de visión: **una carpeta = una
clase**.

A continuación cargamos las imágenes y visualizamos una muestra aleatoria de 10 imagenes, cada una de ellas correspondiente a una clase diferente de las 102 disponibles en el dataset.

In [ ]:
def load_caltech_subset(subset_path):
    """Load all images from the Caltech-101 subset directory.
    Returns a list of (PIL image, class_name) tuples."""
    image_data = []
    for class_name in sorted(os.listdir(subset_path)):
        class_dir = os.path.join(subset_path, class_name)
        if not os.path.isdir(class_dir):
            continue
        for file in os.listdir(class_dir):
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                img = Image.open(os.path.join(class_dir, file)).convert('RGB')
                image_data.append((img, class_name))
    return image_data

def get_sample(image_data, n=10, seed=None):
    """Randomly select n images from image_data.
    Returns two lists: PIL images and class names."""
    if seed is not None:
        random.seed(seed)
    selected = random.sample(image_data, min(n, len(image_data)))
    images = [img for img, _ in selected]
    classes = [cls for _, cls in selected]
    return images, classes

def show_sample(images, classes):
    """Display images with their IDs and class names."""
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i < len(images):
            ax.imshow(images[i])
            ax.set_title(f"ID: {i}\n{classes[i]}", fontsize=8)
            ax.axis('off')
        else:
            ax.axis('off')
    plt.suptitle("Caltech-101 subset: random sample", fontweight='bold')
    plt.tight_layout()
    plt.show()


# Load dataset and show a random sample
subset_path = './Caltech101_subset/Caltech101_subset/'
image_data = load_caltech_subset(subset_path)

print(f"Total images loaded: {len(image_data)}")
print(f"Classes found: {len(set(c for _, c in image_data))}")

sample_images, sample_classes = get_sample(image_data)
show_sample(sample_images, sample_classes)


Antes de pasarle las imágenes al modelo, necesitamos preprocesarlas:
redimensionar a 224×224, recortar el centro y normalizar los valores de
píxel. Esto es lo que hace la función `preprocess` que cargamos junto al
modelo.

A continuación podéis ver el efecto del preprocesado comparando cada
imagen original con su versión transformada.

In [ ]:
def preprocess_images(images):
      """Apply CLIP preprocess to a list of PIL images. Returns a list of tensors."""
      preprocessed = []
      for img in images:
          # pista:En algún punto anterior hemos visto la función que necesitamos. Esta aplica resize, crop y normalización, devuelve tensor [3, 224, 224]
          tensor = ...   # <-- COMPLETE HERE
          preprocessed.append(tensor)
      return preprocessed


In [ ]:
def show_original_vs_preprocessed(images, classes, preprocessed):
    """Display original and preprocessed images side by side."""
    fig, axes = plt.subplots(2, len(images), figsize=(20, 5))

    for i, (img, class_name, tensor) in enumerate(zip(images, classes, preprocessed)):
        axes[0, i].imshow(img)
        axes[0, i].set_title(f"ID: {i}\n{class_name}", fontsize=7)
        axes[0, i].axis('off')

        img_vis = tensor.permute(1, 2, 0).numpy()
        img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min())
        axes[1, i].imshow(img_vis)
        axes[1, i].set_title("preprocessed", fontsize=7)
        axes[1, i].axis('off')

    axes[0, 0].set_ylabel("Original", fontsize=9)
    axes[1, 0].set_ylabel("Preprocessed", fontsize=9)
    plt.suptitle("Original vs Preprocessed images", fontweight='bold')
    plt.tight_layout()
    plt.show()

sample_preprocessed = preprocess_images(sample_images)
show_original_vs_preprocessed(sample_images, sample_classes, sample_preprocessed)


### Preguntas sobre el preprocesador de imágenes

Como hemos dicho antes, el preprocesador se encarga de estandarizar las imágenes antes de pasarlas al encoder para que este reciba los datos en el formato esperado.

---

**P4: ¿Hay algo que ya puedas apreciar de manera visual después de el preprocesamiento?**

*Escribe tu respuesta aquí*

---

## El truco de magia

Antes de explicar nada técnicamente, vamos a ver CLIP en acción. Para ello usaremos el sample que has obtenido justo arriba. Para ello, voy a pedirte que describas cada una de las imagenes anteriores en una frase en *inglés*.

Ej: "*This is a turtle swimming in the ocean*",
    "*This is a big boat, maybe a ferry*",
    "*This is some feline, like a big cat*"

Para mejorar todavía más el ejercicio trata de no gastar la palabra si puedes que describe la clase. Por ejemplo:

En lugar de "scissors" ----> "This is a tool used to cut paper"


In [ ]:
description = [None]*10
# COMPLETE HERE
description[0] = "This is a big mammal"
description[1] = ""
description[2] = ""
description[3] = ""
description[4] = ""
description[5] = ""
description[6] = ""
description[7] = ""
description[8] = ""
description[9] = ""
# END OF COMPLETE HERE

Ya tenemos todo listo para hacer nuestra primera comparación texto-imagen
con CLIP.

El proceso es el siguiente:

1. **Codificamos las imágenes** — pasamos los tensores preprocesados por
`model.encode_image()`, que devuelve un vector de 512 dimensiones por
imagen
2. **Codificamos los textos** — pasamos las descripciones por
`model.encode_text()`, que devuelve también un vector de 512 dimensiones
por texto
3. **Calculamos la similitud** — multiplicamos ambas matrices para obtener
  cuánto se parece cada imagen a cada texto

`torch.no_grad()` indica que no necesitamos calcular gradientes — no
estamos entrenando nada, solo haciendo inferencia. Esto ahorra memoria y
acelera el cálculo.


In [ ]:
# Input para Encode images: stack preprocessed tensors into a single batch and move to device (GPU)
# torch.stack() agrupa la lista de tensores [3,224,224]   en un único tensor [N,3,224,224]
# .to(device) lo mueve a GPU si está disponible
image_input = ... # <-- COMPLETE HERE

# Encode texts: tokenize the descriptions and move to device (GPU)
# tokenizer_clip convierte cada frase en una secuenciade 77 IDs numéricos
# .to(device) lo mueve a GPU para que esté en el mismo  sitio que el modelo
text_tokens = ... # <-- COMPLETE HERE


# Run the model — no gradients needed since we are not training
# encode_image pasa cada imagen por el  ViT y devuelve un vector de 512 dims
# encode_text pasa cada frase por el  transformer y devuelve un vector de 512 dims
with torch.no_grad():
    image_features = ...  # <-- COMPLETE HERE
    text_features  = ...  # <-- COMPLETE HERE

Con los embeddings de imágenes y textos calculados, ya podemos medir
cuánto se parece cada imagen a cada descripción. Usamos el **producto
matricial** (en python `@`) entre ambas matrices — equivalente a calcular la
similitud coseno entre todos los pares posibles a la vez.

El resultado es una matriz de forma `[N_imágenes × N_textos]` donde cada
celda indica cuánto se parece la imagen `i` al texto `j`.

Aplicamos `softmax` para convertir las similitudes en probabilidades que
sumen 1 por fila — así es más fácil interpretar qué texto describe mejor
cada imagen.

In [ ]:
# Compute similarity matrix between image and text embeddings
# image_features shape: [N_images, 512]
# text_features shape:  [N_texts, 512]
# pista: [N_images, 512] @ [N_texts, 512] es producto matricial entre imposible, hay que transponer algo
# .softmax(dim=-1) — normaliza cada fila para que sume 1
# .cpu() — mueve el resultado de GPU a CPU para poder visualizarlo
similarity = ...  # <-- COMPLETE HERE


In [ ]:
def show_similarity_matrix(images, texts, similarity):
      """Display cosine similarity matrix with images on x-axis and texts on y-axis."""
      fig, ax = plt.subplots(figsize=(20, 14))
      ax.imshow(similarity, vmin=0.1, vmax=0.3)
      ax.set_yticks(range(len(texts)))
      ax.set_yticklabels(texts, fontsize=18)
      ax.set_xticks([])

      for i, tensor in enumerate(images):
          img_vis = tensor.permute(1, 2, 0).numpy()
          img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min())
          ax.imshow(img_vis, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

      for x in range(similarity.shape[1]):
          for y in range(similarity.shape[0]):
              ax.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

      for side in ["left", "top", "right", "bottom"]:
          ax.spines[side].set_visible(False)

      ax.set_xlim([-0.5, len(images) - 0.5])
      ax.set_ylim([len(texts) + 0.5, -2])
      ax.set_title("Cosine similarity between text and image features",  size=20)
      plt.tight_layout()
      plt.show()

show_similarity_matrix(sample_preprocessed, description, similarity)

---
## Parte I: Imágenes como datos

### Fundamento teórico

Para un ordenador, una imagen es simplemente una **matriz de números**. Cada píxel tiene tres valores entre 0 y 255, uno por canal: **R**ojo, **V**erde, **A**zul (RGB).

Pero esta representación no tiene semántica: dos fotos de perros distintos son matrices completamente diferentes. Necesitamos un **encoder** que entienda el contenido visual, igual que el tokenizador + modelo de lenguaje entiende el texto.

Antes de llegar ahí, vamos a entender cómo se divide la imagen en unidades procesables — la equivalencia visual de la tokenización.

### La analogía fundamental

| Texto | Imagen |
|---|---|
| `"hola mundo"` | foto de un gato |
| tokens: `["hola", "mundo"]` | patches: `[parche₁, parche₂, ...]` |
| embedding por token | embedding por patch |
| `[CLS]` token resumen global | class token resumen global |

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Patchification.png" width="800">
</div>

### Ejemplo guiado: la imagen como datos numéricos

Ejecuta la celda y observa:

- El shape de la imagen → alto × ancho × canales de color (R, G, B)
- Los valores de píxel van de 0 a 255
- Los tres canales separados: cada uno captura la intensidad de un color

> Esto es lo que recibe el modelo antes de hacer cualquier cosa inteligente. La magia empieza después.


In [ ]:
def get_images_by_class(image_data, class_name, n=2):
    """Return up to n PIL images from a given class."""
    return [img for img, cls in image_data if cls == class_name][:n]

def encode_images(images):
    """Encode a list of PIL images with CLIP. Returns normalized numpy array (N, 512)."""
    image_input = torch.stack(preprocess_images(images)).to(device)
    with torch.no_grad():
        features = model.encode_image(image_input).float()
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy()

def encode_texts(texts):
    """Encode a list of strings with CLIP. Returns normalized numpy array (N, 512)."""
    text_tokens = tokenizer(texts).to(device)
    with torch.no_grad():
        features = model.encode_text(text_tokens).float()
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy()

print('Helper functions defined: get_images_by_class, encode_images, encode_texts')


In [ ]:
# GUIDED EXAMPLE — run and observe

sample_img = sample_images[0]
img_array = np.array(sample_img)  # shape (height, width, 3)

print(f"Image shape: {img_array.shape}  → (height, width, channels)")
print(f"Pixel values range: {img_array.min()} to {img_array.max()}")
print(f"Class: {sample_classes[0]}")
print(f"\nTop-left 3x3 pixels (RGB values):")
print(img_array[:3, :3, :])

# Visualize the image and its three RGB channels
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(sample_img)
axes[0].set_title(f"Original ({sample_classes[0]})", fontweight='bold')

channel_names = ['Red channel', 'Green channel', 'Blue channel']
cmaps = ['Reds', 'Greens', 'Blues']
for i, (ax, name, cmap) in enumerate(zip(axes[1:], channel_names, cmaps)):
    channel = img_array[:, :, i]
    ax.imshow(channel, cmap=cmap)
    ax.set_title(name)
    ax.set_xlabel(f"min={channel.min()}, max={channel.max()}")

for ax in axes:
    ax.axis('off')
axes[1].axis('on'); axes[1].set_xticks([]); axes[1].set_yticks([])
axes[2].axis('on'); axes[2].set_xticks([]); axes[2].set_yticks([])
axes[3].axis('on'); axes[3].set_xticks([]); axes[3].set_yticks([])

plt.suptitle('Image seen as numbers: three channels (R, G, B)', fontweight='bold')
plt.tight_layout()
plt.show()


### Ejercicio 1: Patchification — la tokenización visual (Dificultad: Baja)

Los modelos Vision Transformer (ViT) dividen la imagen en **patches** (parches) de tamaño fijo, igual que el tokenizador divide el texto en tokens.

Para que los patches coincidan con lo que el modelo ve realmente, trabajaremos sobre la imagen **preprocesada** (224×224 px). El modelo que usamos es `ViT-B-32`, donde el `32` indica que cada patch mide **32×32 píxeles**.

> **Pregunta antes de programar:** Una imagen de 224×224 píxeles dividida en patches de 32×32, ¿cuántos patches (tokens visuales) tiene?

Completa el código para dividir la imagen en patches y visualizarlos en una cuadrícula.


In [ ]:
def extract_patches(img_array, patch_size):
    """Divide una imagen en patches de patch_size×patch_size. Returns lista de patches."""
    n = img_array.shape[0] // patch_size
    patches = []
    for i in range(n):
        for j in range(n):
            # TODO: Extrae el patch (i, j) de img_array
            # pista: el patch (i,j) empieza en la fila i*patch_size y termina en (i+1) con una diferencia de espacio de patch_size
            # igual para columnas con j
            patch = img_array[
                i*patch_size : ... ,   # <-- COMPLETE filas
                j*patch_size : ... ,   # <-- COMPLETE columnas
                :
            ]
            patches.append(patch)
    return patches, n

def visualize_patches(patches, n, patch_size):
    """Visualiza patches en una cuadrícula n×n."""
    fig, axes = plt.subplots(n, n, figsize=(8, 8))
    for i in range(n):
        for j in range(n):
            axes[i][j].imshow(patches[i*n + j])
            axes[i][j].axis('off')
    plt.suptitle(
        f"Image divided into {n**2} patches of {patch_size}×{patch_size}px\n"
        f"Each patch = one visual 'token' (just like ViT-B-32 sees it)",
        fontweight='bold'
    )
    plt.tight_layout()
    plt.show()


# Usamos la imagen preprocesada (224×224) — lo que realmente ve ViT-B-32
img_array = sample_preprocessed[0].permute(1, 2, 0).numpy()
img_array = (img_array - img_array.min()) / (img_array.max() -
img_array.min())

patch_size = 32  # ViT-B-32: patches de 32×32 píxeles

# TODO: Calcula cuántos patches caben por lado
n_patches_per_side =   # <-- COMPLETE HERE (pista: 224 // 32)

print(f"Image size: 224x224 pixels")
print(f"Patch size: {patch_size}x{patch_size} pixels")
print(f"Patches per side: {n_patches_per_side}")
print(f"Total patches (tokens): {n_patches_per_side**2 if n_patches_per_side else '?'}")

# TODO: Extrae los patches y visualízalos
# pista: usa extract_patches
patches, n = ... # <-- COMPLETE HERE
visualize_patches(patches, n, patch_size)

**Reflexiona:** Cada patch contiene información visual local — el cielo en un patch, un ala en otro, una rueda en otro. Igual que cada token de texto tiene información semántica local. El transformer aprende a relacionar todos los patches entre sí para entender la imagen completa.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Plane_patching.png" width="800">
</div>

### Ejercicio: ¿Qué pasa si mezclamos los patches?

Los tokens de texto tienen un orden que importa: "el perro muerde al hombre" ≠ "el hombre muerde al perro". ¿Pasa lo mismo con los patches visuales?

En este ejercicio vas a **reordenar aleatoriamente** los patches de una imagen y reconstruirla. Verás que para un humano la imagen se vuelve irreconocible — pero el modelo ViT recibe exactamente los mismos patches, solo en distinto orden.

> **Reflexiona antes:** ¿Crees que CLIP daría el mismo embedding a la imagen original y a la versión con patches mezclados? ¿Por qué?

In [ ]:
def reconstruct_from_patches(patches, n):
    """Reconstruye una imagen a partir de una lista de patches en orden n×n."""
    rows = [np.concatenate(patches[i*n:(i+1)*n], axis=1) for i in
range(n)]
    return np.concatenate(rows, axis=0)

def visualize_shuffle_comparison(img_array, shuffled_img_array, n):
    """Muestra la imagen original y la versión con patches mezclados."""
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img_array)
    axes[0].set_title('Original', fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(shuffled_img_array)
    axes[1].set_title('Patches mezclados', fontweight='bold')
    axes[1].axis('off')
    plt.suptitle(f'Los mismos {n*n} patches, distinto orden',
fontweight='bold')
    plt.tight_layout()
    plt.show()


img_array = sample_preprocessed[0].permute(1, 2, 0).numpy()
img_array = (img_array - img_array.min()) / (img_array.max() -
img_array.min())

patches, n = extract_patches(img_array, patch_size=32)

shuffled = patches.copy()
random.shuffle(shuffled)
shuffled_img_array = reconstruct_from_patches(shuffled, n)

visualize_shuffle_comparison(img_array, shuffled_img_array, n)

from PIL import Image as PILImage
orig_emb     = encode_images([PILImage.fromarray((img_array *
255).astype(np.uint8))])
shuffled_emb = encode_images([PILImage.fromarray((shuffled_img_array *
255).astype(np.uint8))])

sim = float((orig_emb @ shuffled_emb.T)[0, 0])
print(f"Cosine similarity (original vs patches mezclados): {sim:.3f}")

### Preguntas sobre el orden de los patches

  Al mezclar los patches y volver a codificar la imagen obtenemos una
  similitud coseno con respecto a la imagen original. Recuerda que la
  similitud coseno vale **1** si dos vectores son idénticos y **0** si no
  tienen nada en común.

  - ¿El valor obtenido está más cerca de 0 o de 1?
  - Sabiendo que los patches son los mismos pero en distinto orden: ¿qué
  crees que está capturando CLIP para que el valor no sea ni 0 ni 1?

  ---

  **P5: A la vista del resultado, ¿dirías que el orden de los patches es
  importante para CLIP? ¿Por qué crees que no es completamente
  irrelevante?**

  *Escribe tu respuesta aquí*



---
## Parte II: Embeddings visuales

### Fundamento teórico

Igual que el modelo de lenguaje que usamos en el primer seminario convertía cada token en un vector de 384 dimensiones, el encoder visual de CLIP convierte una imagen en un **vector de 512 dimensiones**.

Este vector captura el **significado visual** de la imagen, su **semántica**: dos fotos de perros producen vectores cercanos, aunque sean perros diferentes. Una foto de un avión produce un vector lejano.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Visual Encoder.png" width="600">
</div>

### Ejemplo guiado: embedding de una imagen

In [ ]:
# GUIDED EXAMPLE — run and observe

example_img = sample_images[0]
example_emb = encode_images([example_img])  # shape (1, 512)

print(f"Class: {sample_classes[0]}")
print(f"Image embedding shape: {example_emb.shape}")
print(f"Embedding dimension: {example_emb.shape[1]}  (same as CLIP text embeddings)")
print(f"First 10 values: {example_emb[0][:10].round(4)}")
print(f"Norm (after normalization): {np.linalg.norm(example_emb[0]):.4f}")


### Ejercicio 2: Similitud coseno entre imágenes (Dificultad: Baja)

¡El mismo operador que usabas en el Seminario 2 para comparar frases funciona ahora con imágenes!

> **Predice antes de ejecutar:** ¿Qué similitud coseno esperas entre dos fotos de perros distintos? ¿Y entre una foto de perro y una de avión? Apunta tus predicciones.

In [ ]:
# Seleccionamos 6 imágenes representativas: 2 de la misma clase + 4 clases distintas
selected = {
    'airplanes_1': get_images_by_class(image_data, 'airplanes')[0],
    'airplanes_2': get_images_by_class(image_data, 'airplanes')[1],
    'dalmatian':   get_images_by_class(image_data, 'dalmatian')[0],
    'elephant':    get_images_by_class(image_data, 'elephant')[0],
    'flamingo':    get_images_by_class(image_data, 'flamingo')[0],
    'grand_piano': get_images_by_class(image_data, 'grand_piano')[0],
}

names = list(selected.keys())
images = list(selected.values())

# TODO: Codifica todas las imágenes con encode_images()
all_image_embs = ... # <-- COMPLETE HERE  shape: (6, 512)

# TODO: Calcula la matriz de similitud coseno (6×6)
sim_matrix = ...  # <-- COMPLETE HERE  (pista: Multiplicar la matriz por ella misma para comparar)

def plot_image_similarity_matrix(sim_matrix, images, title):
    n = len(images)
    thumb = 0.8

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(sim_matrix, vmin=0.5, vmax=1.0, cmap='RdYlGn')

    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{sim_matrix[i, j]:.3f}",
                    ha='center', va='center', fontsize=10)

    ax.set_xticks([])
    ax.set_yticks([])

    for j, img in enumerate(images):
        ax.imshow(img, extent=(j - thumb/2, j + thumb/2, -0.5 - thumb, -0.5),
                  aspect='auto', origin='lower', zorder=3)

    for i, img in enumerate(images):
        ax.imshow(img, extent=(-0.5 - thumb, -0.5, i - thumb/2, i + thumb/2),
                  aspect='auto', origin='lower', zorder=3)

    ax.set_xlim([-0.5 - thumb - 0.1, n - 0.5])
    ax.set_ylim([n - 0.5, -0.5 - thumb - 0.1])
    ax.set_title(title, fontweight='bold', pad=10)
    plt.tight_layout()
    plt.show()


if sim_matrix is not None and not isinstance(sim_matrix, type(...)):
    plot_image_similarity_matrix(
        sim_matrix, images,
        'Similitud coseno entre embeddings de imágenes'
    )
    print("\nObservaciones clave:")
    print(f"  airplanes_1 vs airplanes_2: {sim_matrix[0,1]:.3f}")
    print(f"  airplanes_1 vs dalmatian:   {sim_matrix[0,2]:.3f}")
    print(f"  airplanes_1 vs elephant:    {sim_matrix[0,3]:.3f}")



### Preguntas sobre la similitud entre imágenes

  Acabas de calcular la similitud coseno entre 6 imágenes en el espacio de embeddings de CLIP.

  ---

  **P6: Observa la matriz. ¿Qué te dice sobre cómo CLIP organiza las imágenes en su espacio vectorial?**

  *Escribe tu respuesta aquí*

  ---

### Ejercicio 3: PCA de embeddings visuales (Dificultad: Media)

Vamos a proyectar los embeddings de 50 imágenes (5 por clase) a 2D con PCA, igual que hacíais con palabras en el Seminario 2. ¿Se agruparán por categoría?

> **Pregunta:** ¿Qué clases esperáis que estén más próximas entre sí en el espacio visual?

In [ ]:
from collections import defaultdict

def get_pca_sample(image_data, n_per_class=5):
    """Return images and labels for a fixed set of visually diverse classes."""
    selected_classes = [
      # Animales grandes (deberían agruparse)
      'elephant', 'rhino', 'kangaroo', 'llama', 'panda',
      # Vehículos (deberían agruparse)
      'airplanes', 'helicopter', 'Motorbikes', 'car_side', 'ferry',
      # Instrumentos musicales (deberían agruparse)
      'grand_piano', 'accordion', 'saxophone', 'electric_guitar', 'euphonium',
      # Caras (muy distintas, outlier claro)
      'Faces',
  ]
    per_class = defaultdict(list)
    for img, cls in image_data:
        if cls in selected_classes and len(per_class[cls]) < n_per_class:
            per_class[cls].append(img)

    images = [img for cls in selected_classes for img in per_class[cls]]
    labels = [cls for cls in selected_classes for _ in per_class[cls]]
    return images, labels

def plot_pca_embeddings(embs_2d, labels, pca):
    """Scatter plot of 2D PCA embeddings colored by class."""
    unique_classes = sorted(set(labels))
    palette = plt.cm.get_cmap('tab10', len(unique_classes))
    fig, ax = plt.subplots(figsize=(12, 8))

    for label, coords in zip(labels, embs_2d):
        c_idx = unique_classes.index(label)
        ax.scatter(coords[0], coords[1],
                   color=palette(c_idx), s=90, alpha=0.85,
                   edgecolors='white', linewidth=0.5, zorder=3)

    for c_idx, cls in enumerate(unique_classes):
        mask = [l == cls for l in labels]
        cx = np.array(embs_2d)[mask, 0].mean()
        cy = np.array(embs_2d)[mask, 1].mean()
        ax.text(cx, cy, cls, fontsize=10, fontweight='bold',
                color=palette(c_idx),
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
    ax.set_title('PCA de embeddings CLIP — Caltech-101', fontweight='bold')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
    print(f"Varianza total explicada por 2 PCs: {pca.explained_variance_ratio_.sum():.1%}")


pca_images, pca_labels = get_pca_sample(image_data)

# TODO: Codifica las 50 imágenes
pca_embs =  ... # <-- COMPLETE HERE  shape: (50, 512)

# TODO: Aplica PCA para reducir a 2 dimensiones
pca     = ...  # <-- COMPLETE HERE  (pista: Crear un objeto PCA con n_components igual a 2)
embs_2d = ...  # <-- COMPLETE HERE  (pista: usa el metodo fit_transform para proyectoar los embedings pca a un espacio 2d)

if embs_2d is not None and not isinstance(embs_2d, type(...)):
    plot_pca_embeddings(embs_2d, pca_labels, pca)


### Preguntas sobre el PCA de embeddings visuales

  Acabas de proyectar los embeddings de CLIP a 2 dimensiones con PCA, igual que hacíais con palabras en el Seminario 2. Ahora cada punto es una imagen y su
  posición refleja su representación en el espacio vectorial de CLIP.

  ---

  **P7: Observa el plot. ¿Los puntos de la misma clase tienden a agruparse? ¿Qué clases están más cerca entre sí y qué tienen en común?**

  *Escribe tu respuesta aquí*

---
## Parte III: El espacio compartido texto ↔ imagen

### Fundamento teórico

En los ejercicios anteriores hemos usado el encoder de imágenes de CLIP. Pero CLIP tiene **dos encoders**:

- **Text Encoder**: texto → vector en espacio compartido (512D)
- **Image Encoder**: imagen → vector en **el mismo** espacio compartido (512D)

CLIP se entrenó con **400 millones de pares (imagen, descripción)** de Internet. El objetivo fue simple: hacer que el vector del texto y el vector de la imagen correspondiente estén cerca. Y hacer que pares incorrectos estén lejos.

El resultado: la **similitud coseno entre un texto y una imagen** funciona exactamente igual que entre dos textos.

---
## Parte II: Embeddings visuales

### Fundamento teórico

Igual que el modelo de lenguaje que usamos en el primer seminario convertía cada token en un vector de 384 dimensiones, el encoder visual de CLIP convierte una imagen en un **vector de 512 dimensiones**.

Este vector captura el **significado visual** de la imagen, su **semántica**: dos fotos de perros producen vectores cercanos, aunque sean perros diferentes. Una foto de un avión produce un vector lejano.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Dual_Encoder.png" width="600">
</div>

### Ejemplo guiado: mismo espacio, dos modalidades

In [ ]:
# GUIDED EXAMPLE — run and observe

text = "a photo of an airplane"
airplane_img  = get_images_by_class(image_data, 'airplanes')[0]
elephant_img  = get_images_by_class(image_data, 'elephant')[0]

text_emb     = encode_texts([text])          # shape (1, 512)
airplane_emb = encode_images([airplane_img]) # shape (1, 512)
elephant_emb = encode_images([elephant_img]) # shape (1, 512)

sim_text_airplane = cosine_similarity(text_emb, airplane_emb)[0][0]
sim_text_elephant = cosine_similarity(text_emb, elephant_emb)[0][0]

print(f"Text embedding shape:  {text_emb.shape}")
print(f"Image embedding shape: {airplane_emb.shape}")
print("¡Ambos son 512-dimensionales → viven en el mismo espacio!")
print(f"\nSimilitud coseno:")
print(f"  '{text}' vs airplane: {sim_text_airplane:.4f}")
print(f"  '{text}' vs elephant: {sim_text_elephant:.4f}")


### Ejercicio 4: Matriz de similitud texto ↔ imagen (Dificultad: Media)

Vamos a construir una **matriz cruzada**: filas = imágenes, columnas = textos. Cada celda es la similitud coseno entre ese par.

> **Predice:** ¿Qué patrón esperas ver en la matriz? ¿Dónde estarán los valores más altos?

In [ ]:
# 5 imágenes (una por clase)
cross_classes  = ['airplanes', 'dalmatian', 'elephant', 'flamingo', 'grand_piano']
cross_images   = [get_images_by_class(image_data, c)[0] for c in cross_classes]
cross_texts_en = [
    "a photo of an airplane",
    "a photo of a dalmatian dog",
    "a photo of an elephant",
    "a photo of a flamingo",
    "a photo of a grand piano",
]

# TODO: Codifica todas las imágenes
cross_img_embs = ...     # <-- COMPLETE HERE  shape: (5, 512)

# TODO: Codifica todos los textos en inglés
cross_text_embs_en = ... # <-- COMPLETE HERE  shape: (5, 512)

# TODO: Calcula la matriz de similitud cruzada (textos × imágenes)
cross_sim_en = ...       # <-- COMPLETE HERE

def plot_cross_similarity(sim_matrix, images, text_labels, title):
    n = len(images)
    short_labels = [t.replace("a photo of ", "") for t in text_labels]

    plt.figure(figsize=(8, 7))
    plt.imshow(sim_matrix, vmin=0.1, vmax=0.4, cmap='RdYlGn')
    plt.yticks(range(n), short_labels, fontsize=12)
    plt.xticks([])

    for x in range(n):
        for y in range(n):
            plt.text(x, y, f"{sim_matrix[y, x]:.3f}",
                     ha="center", va="center", size=11, color="black")

    for i, img in enumerate(images):
        plt.imshow(img, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

    for side in ["left", "top", "right", "bottom"]:
        plt.gca().spines[side].set_visible(False)

    plt.xlim([-0.5, n - 0.5])
    plt.ylim([n + 0.5, -2])
    plt.title(title, size=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

if cross_sim_en is not None and not isinstance(cross_sim_en, type(...)):
    plot_cross_similarity(
        cross_sim_en,
        cross_images, cross_texts_en,
        'Similitud cruzada: imágenes × textos en inglés'
    )


### Ejercicio 5: ¿Funciona en castellano? (Dificultad: Baja)

CLIP se entrenó principalmente con textos en inglés. Sin embargo, en el Seminario 2 visteis que los embeddings multilingües pueden mapear idiomas distintos al mismo espacio.

> ¿Creéis que CLIP funcionará con queries en castellano? ¿Por qué sí o por qué no? Discutidlo antes de ejecutar.

In [ ]:
# Mismas imágenes, mismo orden — solo cambian los textos
cross_texts_es = [
    "una foto de un avión",
    "una foto de un perro dálmata",
    "una foto de un elefante",
    "una foto de un flamenco",
    "una foto de un piano de cola",
]

# TODO: Codifica los textos en castellano
cross_text_embs_es = ...  # <-- COMPLETE HERE

# TODO: Calcula la similitud cruzada con textos en castellano
# (usa el mismo cross_img_embs del Ejercicio 4 — las imágenes no cambian)
cross_sim_es = ...  # <-- COMPLETE HERE

if cross_sim_es is not None and not isinstance(cross_sim_es, type(...)):
    plot_cross_similarity(
        cross_sim_es,
        cross_images, cross_texts_es,
        'Similitud cruzada: imágenes × textos en castellano'
    )

    print("\nComparación diagonal (pares texto-imagen correctos):")
    print(f"{'Clase':<15} {'Inglés':>10} {'Castellano':>12} {'Diferencia':>12}")
    print('-' * 52)
    for i, cls in enumerate(cross_classes):
        en = cross_sim_en[i, i]
        es = cross_sim_es[i, i]
        print(f"{cls:<15} {en:>10.3f} {es:>12.3f} {es-en:>+12.3f}")


### Preguntas sobre el espacio multilingüe

  Acabas de repetir el mismo experimento pero con textos en castellano. CLIP se entrenó principalmente con datos en inglés, pero los resultados pueden
  sorprenderte.

  ---

  **P8: Compara la diagonal de ambas matrices (inglés vs castellano). ¿Funciona igual de bien? ¿A qué crees que se debe?**

  *Escribe tu respuesta aquí*

---
## Parte IV: Zero-shot classification y búsqueda semántica

### Fundamento teórico

Una vez que texto e imagen viven en el mismo espacio, la **clasificación** es gratis: compara el embedding de la imagen con los embeddings de los nombres de cada clase y quédate con el más cercano. Sin entrenar nada. Sin ver ningún ejemplo etiquetado. Esto se llama **zero-shot classification**.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Zero-Shot.png" width="900">
</div>

### Ejercicio 6: Zero-shot classification en Caltech-101 (Dificultad: Media)

> **Apuesta:** Sin entrenar absolutamente nada, clasificando solo por similitud coseno con el nombre de la clase, ¿qué *accuracy* esperáis? La línea base aleatoria es ~1% (101 clases). ¿Apostáis más de un 50%?


In [ ]:
# Cargar todas las clases del subset
classes = sorted([d for d in os.listdir(subset_path) if os.path.isdir(os.path.join(subset_path, d))])
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

test_images, test_labels = [], []
for img, cls in image_data:
    test_images.append(img)
    test_labels.append(class_to_idx[cls])

print(f"Imágenes a clasificar: {len(test_images)}")
print(f"Clases: {len(classes)}")

# TODO: Crea un prompt de texto para cada clase y codifícalos
class_prompts = ... # <-- COMPLETE HERE  (pista: usa una lista por comprensión para crear un prompt del tipo "a photo of a {clase}" para cada clase)
class_text_embs = ...  # <-- COMPLETE HERE  shape: (101, 512)

# Codifica todas las imágenes
print("Codificando imágenes...")
all_img_embs = encode_images([img for img, _ in image_data])  # <-- COMPLETE HERE

# TODO: Calcula la matriz de similitud (N_imágenes × 101_clases)
similarity = ... # <-- COMPLETE HERE

# TODO: Predicción = clase con mayor similitud
predicted_labels = ...  # <-- COMPLETE HERE  (pista: argmax(axis=1))

accuracy = (predicted_labels == np.array(test_labels)).mean()

print(f"Zero-shot accuracy en Caltech-101: {accuracy:.1%}")
print(f"Línea base aleatoria: {1/len(classes):.1%}")
print(f"Mejora sobre azar: {accuracy - 1/len(classes):.1%}")


In [ ]:
from sklearn.metrics import confusion_matrix

if accuracy is not None and not isinstance(accuracy, type(...)):
    cm = confusion_matrix(test_labels, predicted_labels)
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm, annot=False, fmt='d', cmap='Blues',
        xticklabels=classes, yticklabels=classes,
        ax=ax, linewidths=0.3
    )
    ax.set_xlabel('Predicted', fontweight='bold')
    ax.set_ylabel('True', fontweight='bold')
    ax.set_title(f'Confusion matrix — zero-shot accuracy: {accuracy:.1%}', fontweight='bold')
    ax.tick_params(axis='x', rotation=90, labelsize=6)
    ax.tick_params(axis='y', labelsize=6)
    plt.tight_layout()
    plt.show()

 ### ¿Qué clasifica bien CLIP y qué no?

  La matriz de confusión da una visión global, pero es difícil de leer con 101 clases. A continuación verás 8 ejemplos individuales: 4 que CLIP ha
  clasificado **correctamente** y 4 que ha **fallado**.

  Para cada imagen se muestra el top-5 de clases más probables según CLIP. Fíjate en los errores: ¿tienen sentido semántico? ¿Confunde clases visualmente
  parecidas o conceptualmente relacionadas?

In [ ]:
import torch.nn.functional as F

if accuracy is not None and not isinstance(accuracy, type(...)):
    # Seleccionar 8 imágenes: 4 bien clasificadas + 4 mal clasificadas
    predicted_labels_arr = np.array(predicted_labels)
    test_labels_arr      = np.array(test_labels)
    correct_idx   = np.where(predicted_labels_arr == test_labels_arr)[0]
    incorrect_idx = np.where(predicted_labels_arr != test_labels_arr)[0]
    selected_idx  = np.concatenate([
        np.random.choice(correct_idx,   min(4, len(correct_idx)),   replace=False),
        np.random.choice(incorrect_idx, min(4, len(incorrect_idx)), replace=False),
    ])

    # Top-5 probabilidades por imagen (softmax sobre similitudes)
    sim_tensor  = torch.tensor(similarity[selected_idx]).float()
    top_probs, top_labels_idx = sim_tensor.softmax(dim=-1).topk(5)

    fig, axes = plt.subplots(len(selected_idx), 2, figsize=(12, 3 * len(selected_idx)))

    for i, img_idx in enumerate(selected_idx):
        # Imagen
        axes[i, 0].imshow(test_images[img_idx])
        true_cls = classes[test_labels_arr[img_idx]]
        pred_cls = classes[predicted_labels_arr[img_idx]]
        color    = 'green' if true_cls == pred_cls else 'red'
        axes[i, 0].set_title(f"True: {true_cls}\nPred: {pred_cls}", color=color, fontsize=9)
        axes[i, 0].axis('off')

        # Barras
        y      = np.arange(5)
        probs  = top_probs[i].numpy()
        labels_bar = [classes[j] for j in top_labels_idx[i].numpy()]
        axes[i, 1].barh(y, probs, color=['green' if l == true_cls else 'steelblue' for l in labels_bar])
        axes[i, 1].set_yticks(y)
        axes[i, 1].set_yticklabels(labels_bar, fontsize=9)
        axes[i, 1].invert_yaxis()
        axes[i, 1].set_xlabel('probability')
        axes[i, 1].grid(axis='x', alpha=0.3)

    plt.suptitle('Top-5 predicciones por imagen (verde = correcto)', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()


### ¿Qué clasifica bien CLIP y qué no?

  La matriz de confusión da una visión global, pero es difícil de leer con 101 clases. A continuación verás 8 ejemplos individuales: 4 que CLIP ha
  clasificado **correctamente** y 4 que ha **fallado**.

  Para cada imagen se muestra el top-5 de clases más probables según CLIP. Fíjate en los errores: ¿tienen sentido semántico? ¿Confunde clases visualmente
  parecidas o conceptualmente relacionadas?

  ---

  **P9: Observa los casos fallidos. ¿Los errores son "razonables" o completamente aleatorios? ¿Qué dice esto sobre cómo CLIP representa el espacio visual?**

  *Escribe tu respuesta aquí*

### Ejercicio 7: El prompt importa (Dificultad: Baja-Media)

La formulación del texto que usamos para representar cada clase afecta al resultado. Vamos a comparar distintas variantes.

> **Experimenta:** Añade tu propia variante de prompt. ¿Puedes mejorar la *accuracy* cambiando solo el texto?

In [ ]:
prompt_templates = [
    "{c}",                       # solo el nombre de la clase
    "a photo of a {c}",          # estándar
    "a picture of a {c}",        # variante
    "an image of a {c}",         # variante
    "un {c}",                    # castellano
    # TODO: Añade tu propio template aquí
    # "{c}",
]

print(f"{'Template':<40} {'Accuracy':>10}")
print('-' * 52)

for template in prompt_templates:
    prompts = [template.format(c=c) for c in classes]

    # TODO: Codifica estos prompts
    tmpl_text_embs = ...  # <-- COMPLETE HERE

    # TODO: Clasifica con este template y calcula accuracy
    tmpl_sim   = ...  # <-- COMPLETE HERE  cosine_similarity(all_img_embs, tmpl_text_embs)
    tmpl_preds = ...  # <-- COMPLETE HERE  tmpl_sim.argmax(axis=1)
    tmpl_acc   = ...  # <-- COMPLETE HERE

    print(f"{template:<40} {tmpl_acc:>10.1%}")


### Preguntas sobre el efecto del prompt

  Acabas de clasificar las mismas imágenes con distintas formulaciones del texto. El modelo, las imágenes y el código son exactamente los mismos — solo
  cambia cómo describes las clases.

  ---

  **P10: ¿Qué template ha dado mejor resultado? ¿Por qué crees que la formulación del texto afecta a la precisión si las imágenes no cambian?**

  *Escribe tu respuesta aquí*

 ---
  # Preguntas de evaluación

  Responde a las siguientes preguntas en este notebook.

  ---

  **Pregunta 1 — Patchification y tokenización**

  Explica la analogía entre la tokenización del texto (Seminario 1) y la *patchification* de imágenes que has visto hoy. ¿Qué papel juega el *class token*
  en ambos casos? ¿Por qué crees que esta arquitectura compartida permite que texto e imagen vivan en el mismo espacio?

  *Escribe tu respuesta aquí*

  ---

  **Pregunta 2 — Espacio compartido y multilingüismo**

  En el Ejercicio 5 comprobaste que CLIP funciona en castellano aunque se entrenó principalmente en inglés. Conecta este resultado con lo que aprendiste
  sobre embeddings multilingües en el Seminario 2. ¿Por qué la similitud coseno entre "un perro" y una foto de un perro es razonablemente alta?

  *Escribe tu respuesta aquí*

  ---

  **Pregunta 3 — Zero-shot y nuevas categorías**

  Zero-shot classification permite clasificar imágenes en categorías que el modelo nunca ha visto etiquetadas. ¿Qué ocurriría si quisieras clasificar
  imágenes de una categoría completamente nueva que no existe en ningún dataset conocido? ¿Qué limitación fundamental tiene este enfoque?

  *Escribe tu respuesta aquí*

  ---

  **Pregunta 4 — Encoders separados**

  CLIP tiene dos encoders completamente independientes: uno para texto y otro para imagen. ¿Por qué crees que se diseñó así en lugar de usar un único modelo
   que recibiera texto e imagen a la vez? ¿Qué ventaja práctica tiene esta separación?

  *Escribe tu respuesta aquí*

